<a href="https://colab.research.google.com/github/shamBITS2024/OCR/blob/main/GLM_OCR_testing_using_huggingface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install vllm nest_asyncio pyngrok uvicorn fastapi


In [ ]:
import nest_asyncio
nest_asyncio.apply()


In [ ]:
from vllm import LLM, SamplingParams

# Initializing a tiny, Colab-safe model
llm = LLM(
    model="Qwen/Qwen3-0.6B", # Or "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
    enforce_eager=True       # Crucial to save Colab VRAM memory
)

sampling_params = SamplingParams(temperature=0.7, max_tokens=100)
outputs = llm.generate(["The future of AI in coding is"], sampling_params)

for output in outputs:
    print(output.outputs[0].text)

# --- Comparison of Cell 3 and Cell 4 ---
# Cell 3 (this cell) is generally better for direct text generation within the notebook.
# It initializes the LLM object in the current process and directly performs inference.
# Cell 4 (YsHTgkToafvR) starts a vLLM server in a separate subprocess. This approach is
# intended for serving the model via an API and would require additional client-side code
# to send requests to that server for text generation from within the notebook.

### GLM-based OCR using GOT-OCR2.0
This model can handle plain text OCR, formatted OCR (JSON/Markdown), and even cropped images. We will use a standard prompt to trigger the OCR behavior.

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

model_name = "zai-org/GLM-OCR"

# Using the specialized Processor and ImageTextToText model classes
processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
).eval()

print(f"GLM-OCR model {model_name} successfully loaded with AutoModelForImageTextToText.")

Now, let's run the OCR. You'll need to provide an image path.

In [ ]:
import os
import time
from PIL import Image
import torch

# Define the image path
image_path = "/content/JK202526001207788_13_20250911.jpg"

# Check if file exists before proceeding
if not os.path.exists(image_path):
    print(f"❌ File not found: {image_path}")
else:
    print(f"🚀 Starting inference on: {image_path}")
    image = Image.open(image_path).convert("RGB")

    # Prepare the message structure
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": "Text Recognition:"}
            ],
        }
    ]

    start_time = time.time()

    with torch.no_grad():
        # Process inputs using chat template
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        ).to(model.device)

        # Remove token_type_ids if present (can cause issues with some vision models)
        inputs.pop("token_type_ids", None)

        # Generate response
        generated_ids = model.generate(**inputs, max_new_tokens=2048)

        # Decode output, skipping the prompt tokens
        output_text = processor.decode(generated_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    end_time = time.time()
    latency = end_time - start_time

    print("\n--- GLM-OCR Result ---")
    print(output_text)
    print(f"\n⏱️ Latency: {latency:.2f} seconds")

In [ ]:
import glob
import time
import os
from PIL import Image
import torch

# Find all images in the content folder
image_files = glob.glob("/content/*.jpg")
print(f"📂 Found {len(image_files)} images for batch processing.")

results = []
total_start_time = time.time()

for idx, img_path in enumerate(image_files):
    print(f"Processing [{idx+1}/{len(image_files)}]: {os.path.basename(img_path)}...")

    try:
        image = Image.open(img_path).convert("RGB")

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": "Text Recognition:"}
                ],
            }
        ]

        with torch.no_grad():
            inputs = processor.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt"
            ).to(model.device)

            inputs.pop("token_type_ids", None)
            generated_ids = model.generate(**inputs, max_new_tokens=2048)
            output_text = processor.decode(generated_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

            results.append({"file": img_path, "text": output_text})

    except Exception as e:
        print(f"❌ Error processing {img_path}: {e}")

total_end_time = time.time()
total_duration = total_end_time - total_start_time
avg_latency = total_duration / len(image_files) if image_files else 0

print(f"\n✅ Batch complete!")
print(f"Total Time: {total_duration:.2f}s")
print(f"Average Latency per image: {avg_latency:.2f}s")

In [ ]:
import glob
import time
from PIL import Image
import torch

# Find all images in the content folder
image_files = glob.glob("/content/*.jpg")
print(f"📂 Found {len(image_files)} images for batch processing.")

results = []
total_start_time = time.time()

for idx, img_path in enumerate(image_files):
    print(f"Processing [{idx+1}/{len(image_files)}]: {os.path.basename(img_path)}...")

    try:
        image = Image.open(img_path).convert("RGB")

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": "Text Recognition:"}
                ],
            }
        ]

        with torch.no_grad():
            inputs = processor.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt"
            ).to(model.device)

            inputs.pop("token_type_ids", None)
            generated_ids = model.generate(**inputs, max_new_tokens=2048)
            output_text = processor.decode(generated_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

            results.append({"file": img_path, "text": output_text})

    except Exception as e:
        print(f"❌ Error processing {img_path}: {e}")

total_end_time = time.time()
avg_latency = (total_end_time - total_start_time) / len(image_files) if image_files else 0

print(f"\n✅ Batch complete!")
print(f"Total Time: {total_end_time - total_start_time:.2f}s")
print(f"Average Latency per image: {avg_latency:.2f}s")

In [ ]:
import torch
import time
import os
from PIL import Image

BATCH_SIZE = 2  # Adjust based on VRAM (try 2 or 4 for T4 GPU)
image_files = [f for f in image_files if os.path.exists(f)]

print(f"🚀 Starting Tensor Batching with Batch Size: {BATCH_SIZE}")

batched_results = []
total_start_time = time.time()

for i in range(0, len(image_files), BATCH_SIZE):
    batch_paths = image_files[i : i + BATCH_SIZE]
    batch_images = [Image.open(p).convert("RGB") for p in batch_paths]

    print(f"Processing batch {i//BATCH_SIZE + 1}... images: {[os.path.basename(p) for p in batch_paths]}")

    # Prepare messages for the whole batch
    batch_messages = [
        [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": "Text Recognition:"}
                ],
            }
        ] for img in batch_images
    ]

    with torch.no_grad():
        # Apply chat template and return tensors
        inputs = processor.apply_chat_template(
            batch_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            padding=True, # Padding is required for batching
            return_tensors="pt"
        ).to(model.device)

        inputs.pop("token_type_ids", None)

        # Generate for the entire batch
        generated_ids = model.generate(**inputs, max_new_tokens=1024)

        # Decode each result in the batch
        prompt_len = inputs["input_ids"].shape[1]
        for j, g_ids in enumerate(generated_ids):
            output_text = processor.decode(g_ids[prompt_len:], skip_special_tokens=True)
            batched_results.append({"file": batch_paths[j], "text": output_text})

total_duration = time.time() - total_start_time
avg_latency = total_duration / len(image_files) if image_files else 0

print(f"\n✅ Tensor Batching Complete!")
print(f"Total Time: {total_duration:.2f}s")
print(f"Average Latency per image (Batched): {avg_latency:.2f}s")

In [ ]:
import time
import os
from PIL import Image
import torch

# 1. Sequential Processing Baseline
print("🏃 Starting Sequential Processing...")
sequential_results = []
seq_start = time.time()

for img_path in image_files:
    image = Image.open(img_path).convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": "Text Recognition:"}]}]

    with torch.no_grad():
        inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
        inputs.pop("token_type_ids", None)
        generated_ids = model.generate(**inputs, max_new_tokens=1024)
        output_text = processor.decode(generated_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        sequential_results.append(output_text)

seq_total = time.time() - seq_start
print(f"✅ Sequential Total Time: {seq_total:.2f}s (Avg: {seq_total/len(image_files):.2f}s/img)")

In [ ]:
# 2. Batch Size = 4
BATCH_SIZE_4 = 4
print(f"\n🚀 Starting Batch Processing (Size={BATCH_SIZE_4})...")
batch4_results = []
b4_start = time.time()

for i in range(0, len(image_files), BATCH_SIZE_4):
    batch_paths = image_files[i : i + BATCH_SIZE_4]
    batch_images = [Image.open(p).convert("RGB") for p in batch_paths]
    batch_messages = [[{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": "Text Recognition:"}]}] for img in batch_images]

    with torch.no_grad():
        inputs = processor.apply_chat_template(batch_messages, tokenize=True, add_generation_prompt=True, return_dict=True, padding=True, return_tensors="pt").to(model.device)
        inputs.pop("token_type_ids", None)
        generated_ids = model.generate(**inputs, max_new_tokens=1024)

        prompt_len = inputs["input_ids"].shape[1]
        for g_ids in generated_ids:
            batch4_results.append(processor.decode(g_ids[prompt_len:], skip_special_tokens=True))

b4_total = time.time() - b4_start
print(f"✅ Batch=4 Total Time: {b4_total:.2f}s (Avg: {b4_total/len(image_files):.2f}s/img)")

print(f"\n📊 Speedup: {seq_total / b4_total:.2f}x faster than sequential")

In [ ]:
import time
import os
from PIL import Image
import torch

print(f"🔍 Analyzing individual image performance for {len(image_files)} files...\n")

individual_stats = []

for img_path in image_files:
    filename = os.path.basename(img_path)
    image = Image.open(img_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": "Text Recognition:"}
            ],
        }
    ]

    start = time.time()
    with torch.no_grad():
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        ).to(model.device)

        inputs.pop("token_type_ids", None)
        generated_ids = model.generate(**inputs, max_new_tokens=1024)
        output_text = processor.decode(generated_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    duration = time.time() - start
    char_count = len(output_text)

    print(f"📄 File: {filename}")
    print(f"   ⏱️ Time: {duration:.2f}s")
    print(f"   📝 Output Length: {char_count} characters")
    print("-" * 30)

    individual_stats.append({"file": filename, "time": duration, "len": char_count})

### Structured Extraction Test
We are now moving from full text recognition to targeted attribute extraction.
**Fields to extract:** Certificate ID, Name, Father's Name, Amount, and Date.

In [ ]:
import torch
import os
from PIL import Image

# Defining the targeted prompt
structured_prompt = """Extract the following information from the image in a clear list format:
- Certificate ID
- Name
- Father's Name
- Amount
- Date"""

print("🎯 Starting Structured Extraction...\n")

structured_results = []

# We will test on a subset or all images
for img_path in image_files[:3]:  # Testing on the first 3 for speed
    filename = os.path.basename(img_path)
    image = Image.open(img_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": structured_prompt}
            ],
        }
    ]

    with torch.no_grad():
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        ).to(model.device)

        inputs.pop("token_type_ids", None)
        generated_ids = model.generate(**inputs, max_new_tokens=512)
        output_text = processor.decode(generated_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    print(f"📄 File: {filename}")
    print(f"Extracted Data:\n{output_text}")
    print("-" * 30)

    structured_results.append({"file": filename, "data": output_text})

### JSON-Enforced Extraction
By explicitly requesting a JSON structure and providing a template, we can significantly improve the model's reliability for downstream data processing.

In [ ]:
import json
import re

json_prompt = """Extract the following information from the image and return it ONLY as a valid JSON object.
Fields: Certificate ID, Name, Father's Name, Amount, Date.

Format the output like this:
{
  \"certificate_id\": \"string or null\",
  \"name\": \"string or null\",
  \"fathers_name\": \"string or null\",
  \"amount\": \"string or null\",
  \"date\": \"string or null\"
}"""

print("💎 Starting JSON-Enforced Extraction...\n")

final_json_results = []

for img_path in image_files[:3]:
    filename = os.path.basename(img_path)
    image = Image.open(img_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": json_prompt}
            ],
        }
    ]

    with torch.no_grad():
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        ).to(model.device)

        inputs.pop("token_type_ids", None)
        generated_ids = model.generate(**inputs, max_new_tokens=512)
        output_text = processor.decode(generated_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    # Attempt to parse JSON
    try:
        # Clean output in case model wraps in markdown blocks
        clean_text = re.search(r'\{.*\}', output_text, re.DOTALL)
        if clean_text:
            parsed_data = json.loads(clean_text.group())
        else:
            parsed_data = json.loads(output_text)
    except Exception:
        parsed_data = {"raw_error": "Failed to parse", "content": output_text}

    print(f"📄 File: {filename}")
    print(json.dumps(parsed_data, indent=2))
    print("-" * 30)

    final_json_results.append({"file": filename, **parsed_data})

### Exporting Results
We will now convert the extracted JSON data into a CSV file for easier analysis and sharing.

In [ ]:
import pandas as pd

# Create a DataFrame from the structured results
df_results = pd.DataFrame(final_json_results)

# Save to CSV
csv_filename = "ocr_extraction_results.csv"
df_results.to_csv(csv_filename, index=False)

print(f"✅ Results successfully exported to {csv_filename}")

# Display the first few rows
df_results.head()

### Download Extraction Results
Run the cell below to download the CSV file to your computer.

In [ ]:
from google.colab import files

try:
    files.download(csv_filename)
except Exception as e:
    print(f"Could not trigger automatic download: {e}")
    print(f"You can manually download the file '{csv_filename}' from the file browser on the left.")